In [8]:
# Import Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [9]:
df = pd.read_csv("cleaned_data.csv")

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1997 entries, 0 to 1996
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Unnamed: 0  1997 non-null   int64
 1   ID          1997 non-null   int64
 2   title       1997 non-null   str  
 3   tags        1997 non-null   str  
dtypes: int64(2), str(2)
memory usage: 168.9 KB


In [11]:
# Check Total Vocabulary Size
temp_vectorizer = TfidfVectorizer(stop_words="english")

temp_vectorizer.fit(df["tags"])

print("Total Unique Words:", len(temp_vectorizer.vocabulary_))

Total Unique Words: 1867


In [12]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(
    max_features=10000,
    stop_words="english"
)

vectors = tfidf.fit_transform(df["tags"]).toarray()

vectors.shape

(1997, 1867)

In [13]:
# Cosine Similarity
similarity = cosine_similarity(vectors)

similarity.shape

(1997, 1997)

In [14]:
# Function to Get Book Name from Index

def get_name_by_index(index):

    if index >= 0 and index < len(df):
        return df.loc[index, "title"]

    return "Book Not Found"

In [15]:
# Function to Get Index from Book Name

def get_index_from_name(book_name):

    clean_name = book_name.strip().lower()

    match = df[df["title"].str.lower() == clean_name]

    if not match.empty:
        return match.index[0]

    return -1

In [16]:
# Recommendation Function

def recommend(book_name):

    index = get_index_from_name(book_name)

    if index == -1:
        print("Book Not Found")
        return

    similarity_scores = list(enumerate(similarity[index]))

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    print("\nRecommended Books:\n")

    for i in similarity_scores[1:6]:

        print(get_name_by_index(i[0]))

In [17]:
# Test Recommendation

recommend("The Four Loves")


Recommended Books:

The Problem of Pain
The voyage of the Dawn Treader
The Screwtape Letters ; With, Screwtape Proposes a Toast
The Chronicles of Narnia: Lion, the witch and the wardrobe
Purpose Driven Life MM Camouflage Edition


In [18]:
pickle.dump(similarity, open("similarity.pkl", "wb"))

pickle.dump(tfidf, open("tfidf.pkl", "wb"))